# Stock Price Forecasting — A Baseline ML Study
### Linear Regression on Historical Closing Prices (NSE India & Global Markets)

## What This Project Does
This tool fetches 5 years of real historical data for any stock listed on NSE (India) 
or major global exchanges, trains a Linear Regression model on historical closing prices, 
and produces a single predicted price N days into the future along with a BUY/HOLD/SELL signal.

## Methodology
- **Data:** 5 years of daily closing prices fetched live from Yahoo Finance via yfinance
- **Features:** The closing price plus N lagged closing prices (where N = the user's chosen forecast horizon)
- **Target variable:** The closing price exactly N trading days in the future
- **Train/test split:** Temporal — first 80% of data for training, last 20% for testing. 
  No shuffling, ensuring the model is always tested on genuinely future data it never saw during training.
- **Model:** Linear Regression — chosen for interpretability and as a transparent baseline
- **Evaluation:** Mean Absolute Error (MAE) compared against a naive baseline 
  ("future price = today's price"). The naive baseline serves as the primary benchmark for evaluating whether
  the model adds predictive value beyond simple price persistence.

## Limitations
- **Price persistence:** Stock prices are highly autocorrelated — tomorrow's price is almost 
  always close to today's. A significant portion of the model's predictive power comes from 
  this persistence rather than learned patterns. This is a known limitation of price-level 
  forecasting and is why the naive baseline comparison is included.
- **Single prediction point:** The model predicts one price target N days ahead. 
  It does not model the daily path to that target — for trajectory prediction, 
  sequence models such as LSTM are more appropriate.
- **Features:** The current version uses only historical closing prices. 
  A more robust model would incorporate volume, volatility, technical indicators 
  (RSI, MACD, moving averages), and macroeconomic factors.
- **No guarantee:** This tool is for educational and research purposes only. 
  Stock prices are influenced by news, sentiment, and global events that no model 
  trained on historical prices alone can anticipate.

## Tech Stack
- Python
- pandas
- numpy
- scikit-learn
- yfinance
- matplotlib
- Jupyter Notebook

## Version
This is Version 1 — price level prediction using lagged closing prices.  
Version 2 (in progress) restructures the problem as return prediction and 
includes a full backtest comparing model-driven trading against buy-and-hold.

## Sample Results — 30 Day Forecast Horizon

| Stock    | Market    | Model MAE | Naive MAE | Beat Baseline? |
|----------|-----------|-----------|-----------|----------------|
| RELIANCE | NSE India | ₹63.17    | ₹70.78    | ✅ Yes        |
| TCS      | NSE India | ₹251.36   | ₹207.59   | ❌ No         |
| INFY     | NSE India | ₹108.42   | ₹105.69   | ❌ No         |
| AAPL     | NASDAQ    | \$22.39   | \$18.69   | ❌ No         |
| TSLA     | NASDAQ    | \$68.25   | \$39.42   | ❌ No         |
| GOOGL    | NASDAQ    | \$37.59   | \$33.78   | ❌ No         |

*Results reflect model trained on 5 years of historical data with temporal 
train/test split. MAE reported on unseen test period only (last 20% of data).*

Overall Baseline Success Rate:
1 out of 6 stocks (16.7%) beat the naive forecast.

### Key Finding
The model outperforms the naive baseline on only 1 of 6 stocks tested (16.7%). 
Performance varied significantly across stocks, with no consistent pattern 
across markets or sectors.

The most likely explanation is the known limitation of price-level forecasting: 
the model learns long-run price persistence and upward trend from training data, 
but cannot adapt when the test period exhibits different behaviour. Whether that 
difference is driven by sector conditions, macroeconomic events, or simply 
idiosyncratic stock volatility cannot be determined from this analysis alone.

This directly motivates Version 2, which restructures the problem as return 
prediction using technical indicators and evaluates performance via strategy 
backtesting rather than MAE alone.

## Future Work

Version 2 will:
- Predict future returns rather than price levels
- Incorporate technical indicators such as RSI, MACD, moving averages, momentum, and volatility
- Evaluate directional accuracy in addition to prediction error
- Generate trading signals and perform strategy backtesting
- Compare performance against buy-and-hold benchmarks
- Compare multiple models (Linear Regression, Random Forest, XGBoost) under the same backtesting framework

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import numpy as np

In [ ]:
# Checking for valid market input
while True:
    market = input("What market do you want to check? Type 'IN' for India (NSE) or 'GLOBAL' for global stocks: ").upper().strip()
    if market == "IN" or market == "GLOBAL":
        break
    else:
        print("Invalid input. Please type IN or GLOBAL only.")

# Asking the user what stock they want to analyse
ticker_input = input("Enter stock name (e.g. RELIANCE, TCS, AAPL): ").upper().strip()

# Adding .NS for Indian stocks due to Yahoo Finance database structuring
if market == "IN":
    ticker = ticker_input + ".NS"
    currency = "₹"
else:
    ticker = ticker_input
    currency = "$"

# Asking user for prediction range
while True:
    try:
        forecast_days = int(input("How many days ahead do you want to predict? (7, 14, 30, 60, 90): ").strip())
        if forecast_days in [7, 14, 30, 60, 90]:
            break
        else:
            print("Please enter one of: 7, 14, 30, 60, 90")
    except ValueError:
        print("Please enter a valid number.")

print(f"Fetching data for: {ticker_input}")
print(f"Prediction horizon: {forecast_days} days")

In [ ]:
# Downloading 5 years of historical stock data
df = yf.download(ticker, period="5y", auto_adjust=True)

# Checking if the data loaded correctly
print(f"Downloaded {len(df)} days of data for {ticker_input}")
print(df.tail())

In [ ]:
# Removing any incomplete rows at the end of the dataset
while df["Close"].iloc[-1].isna().any():
    df = df.iloc[:-1]

# Creating a copy for training so we dont modify the original data
df_model = df.copy()

# Creating target column - closing price shifted forecast_days forward
df_model["Prediction"] = df_model["Close"].shift(-forecast_days)

# Creating lag columns equal to forecast_days
for i in range(1, forecast_days + 1):
    df_model[f"Close_lag_{i}"] = df_model["Close"].shift(i)

# Dropping rows with no prediction value
df_model.dropna(inplace=True)

print(f"Data shape after preparation: {df_model.shape}")
print(f"Number of features used: {forecast_days}")
print(df_model[["Close", "Close_lag_1", f"Close_lag_{forecast_days}","Prediction"]].tail())

In [ ]:
# Using Close plus all forecast_days lag columns as features
feature_columns = ["Close"] + [f"Close_lag_{i}" for i in range(1, forecast_days + 1)]
X = df_model[feature_columns].values
y = df_model["Prediction"].values

# Temporal split - 80% train, 20% test, NO shuffling
# Everything before the cutoff is training, everything after is testing
split_index = int(len(df_model) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

print(f"Training on {split_index} days ending at: {df_model.index[split_index-1].date()}")
print(f"Testing on {len(X_test)} days starting at: {df_model.index[split_index].date()}")

# Creating and training the model
model = LinearRegression()
model.fit(X_train, y_train)

# Checking accuracy with R² score
r2_score = model.score(X_test, y_test)

# Mean Absolute Error - average rupee difference between prediction and actual
test_predictions = model.predict(X_test)
mae = np.mean(np.abs(test_predictions - y_test))

# Naive baseline - "price in forecast_days = todays price"
naive_predictions = X_test[:, 0]
naive_mae = np.mean(np.abs(naive_predictions - y_test))

print(f"\nModel R² score: {round(r2_score * 100, 2)}%")
print(f"Model MAE: {currency}{round(mae, 2)} average error per prediction")
print(f"Naive baseline MAE: {currency}{round(naive_mae, 2)}")
print(f"Model beats naive baseline: {mae < naive_mae}")

In [ ]:
# Getting the last forecast_days+1 closing prices as features
last_n = df["Close"].tail(forecast_days + 1).values.flatten()
today_features = last_n.reshape(1, -1)
predicted_price = model.predict(today_features)
current_price = round(float(df["Close"].iloc[-1].values[0]), 2)
predicted_price_value = round(float(predicted_price[0]), 2)

# Calculating percentage change
pct_change = round(((predicted_price_value - current_price) / current_price) * 100, 2)

# Trend signal based on percentage change
if pct_change > 2:
    signal = "BUY"
    signal_reason = "model expects meaningful upside"
elif pct_change < -2:
    signal = "SELL"
    signal_reason = "model expects meaningful downside"
else:
    signal = "HOLD"
    signal_reason = "model expects price to remain relatively stable"

print(f"Current {ticker_input} price: {currency}{current_price}")
print(f"Predicted price in {forecast_days} days: {currency}{predicted_price_value}")
print(f"Expected change: {pct_change}%")
print(f"Signal: {signal} — {signal_reason}")
print(f"\nModel MAE: {currency}{round(mae, 2)} | Beats naive baseline: {mae < naive_mae}")

In [ ]:
# Generating predictions on test data only - genuine unseen predictions
feature_columns = ["Close"] + [f"Close_lag_{i}" for i in range(1, forecast_days + 1)]
test_start_date = df_model.index[split_index]
df_model_test = df_model[df_model.index >= test_start_date]
test_predictions_plot = model.predict(df_model_test[feature_columns].values)

# Filling the gap between last model date and today
gap_df = df[df.index > df_model.index[-1]]
gap_dates = gap_df.index
gap_predictions = []
for i in range(len(gap_df)):
    end_idx = df.index.get_loc(gap_dates[i])
    input_prices = df["Close"].iloc[end_idx-forecast_days:end_idx+1].values.flatten()
    if len(input_prices) == forecast_days + 1:
        input_features = input_prices.reshape(1, -1)
        gap_predictions.append(model.predict(input_features)[0])

# Single honest prediction point
last_actual_date = df.index[-1]
predicted_date = pd.bdate_range(start=last_actual_date, periods=forecast_days + 1)[-1]
predicted_price_value = float(predicted_price[0])

# Setting up two charts stacked vertically
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# --- Chart 1: Full 5 year history ---
ax1.plot(df.index, df["Close"], color="blue", linewidth=1, label="Actual Price")
ax1.plot(df_model_test.index, test_predictions_plot, color="red", linewidth=1,
         linestyle="--", label="Model Predicted Price (test period only)")
ax1.plot(gap_dates[:len(gap_predictions)], gap_predictions, color="red", linewidth=1, linestyle="--")
ax1.plot([last_actual_date, predicted_date], [current_price, predicted_price_value],
         color="gray", linewidth=1, linestyle=":")
ax1.scatter([predicted_date], [predicted_price_value], color="green", s=80, zorder=5,
            label=f"{forecast_days} Day Prediction: {currency}{round(predicted_price_value, 2)}")
ax1.axvline(x=last_actual_date, color="gray", linestyle=":", linewidth=1, label="Today")
ax1.set_title(f"{ticker_input} — Full 5 Year Price History vs Model ({forecast_days}d forecast)", fontsize=14)
ax1.set_xlabel("Date")
ax1.set_ylabel("Price ({currency})")
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Chart 2: Last 6 months ---
cutoff_6m = df.index[-1] - pd.DateOffset(months=6)
df_6m = df[df.index >= cutoff_6m]
df_model_test_6m = df_model_test[df_model_test.index >= cutoff_6m]
test_predictions_6m = model.predict(df_model_test_6m[feature_columns].values)
gap_dates_6m = gap_dates[gap_dates >= cutoff_6m]
gap_predictions_6m = gap_predictions[len(gap_predictions)-len(gap_dates_6m):]

ax2.plot(df_6m.index, df_6m["Close"], color="blue", linewidth=1.5, label="Actual Price")
ax2.plot(df_model_test_6m.index, test_predictions_6m, color="red", linewidth=1.5,
         linestyle="--", label="Model Predicted Price (test period only)")
ax2.plot(gap_dates_6m, gap_predictions_6m, color="red", linewidth=1.5, linestyle="--")
ax2.plot([last_actual_date, predicted_date], [current_price, predicted_price_value],
         color="gray", linewidth=1, linestyle=":")
ax2.scatter([predicted_date], [predicted_price_value], color="green", s=80, zorder=5,
            label=f"{forecast_days} Day Prediction: {currency}{round(predicted_price_value, 2)}")
ax2.axvline(x=last_actual_date, color="gray", linestyle=":", linewidth=1, label="Today")
ax2.set_title(f"{ticker_input} — Last 6 Months Price History vs Model ({forecast_days}d forecast)", fontsize=14)
ax2.set_xlabel("Date")
ax2.set_ylabel("Price ({currency})")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()